# Breaking the Bottlenecks: Diversity, Verification, Self-Proposed Tasks, and Efficiency
*How multi-agent fine-tuning sustains diverse reasoning chains, DeepSeekMath-V2's meta-verification builds trustworthy self-checking, Absolute Zero lets a model propose and solve its own training tasks with zero human-curated data, why the efficiency of all this inference matters just as much as raw capability, and open questions on continual learning and self-created environments*

# What Makes a System an "Agent"?
An agent is a generalization of a plain LLM: it has a **goal**, interacts with an environment, collects feedback, and uses that feedback to correct its own steps. Agents are systems that can direct their own process, use tools, and work toward accomplishing a goal — rather than just producing a single response to a single prompt.

In practice today, a plain LLM often isn't powerful enough on its own to drive a full goal end-to-end, so agentic systems are frequently **hand-built as workflows** — orchestrating LLM calls, verifiers (sometimes LLM-as-judge), tool calls, and search algorithms together. In some domains — coding agents being a clear example — these workflows are increasingly being driven more autonomously, without as much hand-specified structure. Regardless of how much is hand-built versus autonomous, an effective agentic workflow needs the ability to **plan and reason over multiple steps, correct itself when it's going in the wrong direction, and keep improving its own capabilities** — which is the self-improvement thread running through this notebook.


# Two Directions for Future Research
This notebook focuses on two broad, complementary directions:
1. **The self-improvement side** — how to keep the self-improvement loop (test-time scaling, train-time scaling, verification) working well, and how to push it beyond narrow domains like math and coding.
2. **The efficiency side** — how to get more intelligence out of a given amount of compute.

Within the self-improvement side specifically, three open problems stand out:
- **Generalization and diversity:** self-improvement loops still tend to be narrow (math, coding), and reasoning chains generated by a single model tend to lack diversity — limiting how far the loop can be pushed.
- **Robust verification:** building verifiers (or meta-verifiers that check the verifiers themselves) remains genuinely hard, especially in domains where a simple final-answer check isn't available.
- **Breaking the data bottleneck:** the prompts used for train-time scaling are still largely selected statically, by humans — an open question is how a self-improvement loop could learn to pick the right training data itself.


# Multiagent Finetuning: Self-Improvement Through Diverse Reasoning Chains

**Paper:** [arxiv.org/abs/2501.05707](https://arxiv.org/abs/2501.05707)

## The Problem
Self-improvement loops like STaR-style rejection sampling (generate solutions, keep the correct ones, fine-tune, repeat) tend to hit **diminishing returns** — performance plateaus after a handful of rounds. The underlying issue: pre-training data is diverse because it was written by many different humans over a long time, but when a **single** model generates its own training data — even at high sampling temperature — its outputs stay comparatively similar to each other. Without genuine diversity in the reasoning chains being generated, the self-improvement loop runs out of new signal to learn from.

## The Idea: Specialize Multiple Agents Instead of One
Rather than relying on one model to generate all the training data, use **multiple specialized agents**, each fine-tuned somewhat differently, to naturally produce more diverse solutions. Two roles are trained:
- **Generation agents:** produce diverse initial candidate solutions.
- **Critic agents:** evaluate and refine those solutions.


# How the Multi-Agent Loop Works
1. Each of several generation agents (all fine-tuned from the same base model, but on independent data) proposes an **initial answer** to a question.
2. Their answers are **summarized** together (either by another model, or simply by concatenating the responses).
3. A **critic agent** reviews this combined set of answers and critiques them — similar in spirit to single-agent self-critique, but working across multiple different agents' answers rather than just one.
4. Each generation agent produces an **updated answer**, informed by the critique and the summary of what other agents produced.
5. **Majority voting** is applied across the updated answers, and the process can repeat across multiple rounds — functioning somewhat like a structured debate.

## Training the Agents
- **Generation agents:** across iterations, outputs are filtered for agreement with the majority-voted answer, and each generation agent is fine-tuned (via supervised fine-tuning) only on its own correctly-matching outputs — letting each one specialize over time rather than converging toward identical behavior.
- **Critic agents:** trained on a mix of trajectories where an answer starts out correct and gets corrected over the course of the debate — teaching the critic to recognize the contrast between correct and incorrect reasoning.

A simpler, "poor man's" version of this same idea: just using a handful of genuinely **different existing models** (rather than fine-tuning specialized ones) to get diverse responses to the same prompt — something already common in practice.


# Does This Actually Preserve Diversity and Keep Improving?
Measured across fine-tuning iterations, using two metrics: **negative log-likelihood** (a proxy for performance — lower is better) and **embedding dissimilarity** (higher means more diverse responses).

- With **multi-agent** fine-tuning, both open-source models tested continued to **improve across many rounds** of fine-tuning, rather than plateauing.
- Response **diversity stayed high** across rounds, rather than collapsing toward similar outputs — the core problem the method set out to solve.
- With **single-agent** fine-tuning (the STaR-style baseline), accuracy tended to **collapse or stop improving** after a similar number of rounds.
- This pattern held across **three different open-source models** tested on the MATH dataset.
- **Generalization beyond the training domain:** models fine-tuned this way on math also showed improved performance on an adjacent domain (GSM8K) that they weren't directly trained on — suggesting the diversity gained through multi-agent fine-tuning transfers, rather than being narrowly overfit to one dataset.

**The core takeaway:** sustaining a self-improvement loop over many rounds seems to require genuinely diverse reasoning chains — and one concrete way to generate that diversity is through multiple independently-specializing agents, rather than relying on a single model (even at high sampling temperature) to provide enough variety on its own.


# DeepSeekMath-V2: Toward Self-Verifiable Mathematical Reasoning

**Paper:** [arxiv.org/abs/2511.22570](https://arxiv.org/abs/2511.22570)

## The Problem With Final-Answer-Only Rewards
The standard RL setup for math (as covered elsewhere in this series) rewards a model based on whether its **final answer** matches a known ground truth. This works well enough to have driven benchmarks like AIME toward saturation — but it has a real blind spot: **a correct final answer doesn't guarantee correct reasoning.** For domains like **theorem proving**, this limitation is even sharper — proofs don't have a single "final numeric answer" to check against at all; what actually matters is whether each step of a rigorous derivation genuinely follows from the last.

## Why LLM-as-Judge Doesn't Work Well Here
Models trained mostly on quantitative reasoning tend to generate mathematically **invalid proofs** — and when asked to verify their own or others' proofs, they often **incorrectly claim flawed proofs are valid**. Human experts, by contrast, can recognize when a proof step doesn't actually follow logically from the previous one, even without a reference answer to compare against — a form of judgment that plain LLM-as-judge doesn't yet reliably replicate for this kind of rigorous, step-by-step reasoning.


# The Fix: Train a Verifier (and a Meta-Verifier)
Rather than relying on final-answer correctness, DeepSeekMath-V2 has **human experts identify issues in proofs directly, without reference solutions**, and uses that data to train a model specifically to **identify issues in proofs** — a dedicated verifier for proof quality itself, not just final-answer correctness.

## The Full Architecture
- A **generator** model produces candidate proofs.
- A **verifier** model reviews those proofs and identifies specific issues in the reasoning.
- A **meta-verifier** reviews the verifier's own analysis — checking whether the verifier's identified issues actually make sense, essentially verifying the verifier.
- The verifier scores proofs on a continuous scale (roughly 0 to 1) based on this review process.

## The Self-Reinforcing Loop
The verifier is used to improve the generator (via RL, using verifier scores as the reward signal), which then produces **harder proofs** — which in turn helps further improve the verifier, since it now has more challenging cases to learn from. Because the meta-verifier's proof-issue identification can itself be automated once it's learned this skill well, this labeling process doesn't need to permanently rely on ongoing human annotation — the system can bootstrap further verification data on its own over time.

## Results
DeepSeekMath-V2 achieved **gold-level scores on IMO 2025 and CMO 2024**, and a near-perfect **118 out of 120** on Putnam 2024, using scaled test-time compute. These are strong, competition-level results specifically in theorem proving — a domain where final-answer-based reward signals don't apply at all, showing that self-verification is a genuinely viable path forward for reasoning tasks that go beyond simple correct/incorrect final answers.


# Why Meta-Verification Specifically Matters
Verifiers can be fooled — they can assign a passing score to a reasoning chain that's actually flawed, sometimes by essentially fabricating an issue that doesn't really exist, or missing one that does. Meta-verification adds a check on top of the verifier itself: does the verifier's identified issue actually exist in the proof? Does the score it assigned actually follow from that issue? Expert human annotators rate the quality of this verification step directly, and just adding this one additional layer measurably improves overall quality — essentially: reasoning chain → verifier flags issues → meta-verifier checks whether those flagged issues are real.

## Results
Built on top of DeepSeek-V3-based models (using a TRPO-style RL approach, similar to how much of this kind of RL training is built), tested on IMO and CMO-style problems: proof quality (pass@1) continued climbing over **8 iterations** of this self-verification loop. Using **best-of-32** (picking the best of 32 generated proofs), the model reached roughly **42%** proof-score accuracy on the IMO Shortlist 2024 problem set — a meaningfully strong result for this kind of hill-climbing technique. This suggests genuine promise: if you can reliably identify issues in reasoning chains and nudge the model toward correcting them specifically, that alone can meaningfully improve output quality.

## Generalizing This Approach Beyond Math
The recipe generalizes into three ingredients: (1) an LLM-based verifier that can identify issues **without needing a reference solution**, (2) a **meta-verification** layer to catch the verifier's own hallucinated or incorrect issue-flagging, and (3) an incentive for the generator to actually **fix** the issues that get flagged, rather than just being scored on them. This is still fundamentally limited to domains where verification (even imperfect, LLM-based verification) is *possible* in the first place — it doesn't solve verification in domains where checking correctness is inherently much harder.


# Absolute Zero: Can a Model Propose Its Own Training Tasks?

**Paper:** [arxiv.org/abs/2505.03335](https://arxiv.org/abs/2505.03335)

## The Problem: Human-Curated Prompts Are a Bottleneck
Standard training approaches — supervised fine-tuning on human-written reasoning traces, or RL with verifiable rewards on human-curated question-answer pairs — both require domain experts to hand-select the training tasks (math experts for math problems, strong software engineers for coding problems, and so on). As models get more capable, finding enough experts and enough sufficiently hard tasks to keep training them becomes an increasingly real bottleneck.

## The Idea: Let the Model Propose Its Own Tasks
Rather than relying on any external, human-curated prompt set at all, **a single model plays both roles** — it proposes new tasks for itself to solve, and then solves them. This is applied specifically in the **coding** domain, using three task types:
- **Deduction:** given a program and an input, predict the output (trace through execution).
- **Abduction:** given a program and an output, infer a plausible input that would produce it (work backward from a result).
- **Induction:** given a set of input-output examples, synthesize a program that explains them (generalize from partial information).

A code executor serves as the grounded environment: it runs the proposed program to check outputs, giving the whole system a reliable, automated way to verify both the validity of a proposed task and the correctness of a solution — without needing any human-labeled data at all.


# How the Proposer Learns What Tasks to Propose
The **proposer** is rewarded based on **task difficulty**, not just on generating any valid task:
- If every attempt to solve a proposed task **succeeds** (task is trivial) or **every attempt fails** (task is effectively impossible), the proposer gets **zero reward**.
- Otherwise, the reward is **1 minus the average solver success rate** — meaning tasks with a moderate difficulty (the solver succeeds sometimes, fails sometimes) are explicitly favored.

This design deliberately steers the proposer toward tasks that sit right at the edge of what the current model can do — not too easy, not impossibly hard — which is exactly the kind of task that provides a useful learning signal. As the solver gets more capable over training, the proposer is pushed to keep proposing harder tasks to keep earning reward, creating a naturally rising difficulty curve without anyone manually increasing it.

## Keeping Proposed Tasks Valid
Since a model proposing its own tasks could in principle generate nonsense, proposed tasks are validated before entering the training pipeline: programs are actually **executed** to check they run without errors, basic **safety checks** are applied, and — since code execution should be deterministic — outputs are checked for **consistency across repeated runs**. A **buffer** of validated (program, input, output) triplets is maintained, and the proposer can sample from and build on this buffer when generating new tasks, along with being explicitly conditioned on a sample of past generated tasks to help maintain diversity rather than repeatedly proposing very similar ones. Together, this produces a form of **curriculum learning** that evolves naturally as training progresses.


# Results and Emergent Behavior
Despite using **zero human-curated prompts**, this approach reached **state-of-the-art results on coding benchmarks**, outperforming models trained on tens of thousands of expert-written examples.

Some notable patterns observed during training:
- **Task complexity increased over time** — expected, since the proposer is directly incentivized to keep raising difficulty as the solver improves.
- **Diversity of generated programs and solutions improved** when the loop was set up well, rather than collapsing toward repetitive tasks.
- The proposer and solver behave somewhat like two sides of a **game** — mildly adversarial to each other, but both improving as a result of that dynamic, similar in spirit to self-play setups.

**A surprising generalization result:** even though the system only trained on **self-proposed coding tasks**, it also showed strong performance gains on **math benchmarks** — a domain it was never directly trained on. And consistent with a broader pattern seen elsewhere, **larger models saw bigger gains** from this approach than smaller ones.


# A Recurring Pattern Across Papers
This cross-domain generalization result (train on coding, see gains on math) echoes a very similar finding from SWiRL (covered earlier in this notebook series): training a model on multi-step tool use in one domain (e.g., search-based question answering) produced genuine transfer to a completely different domain and tool (e.g., math with a calculator), and vice versa. Across multiple independent lines of work, self-generated synthetic training data appears to improve a model not just on the specific task it was generated for, but on the model's broader reasoning and tool-use ability more generally.

A second recurring pattern: **larger models seem to benefit more** from this kind of self-generated data flywheel — both in Absolute Zero and in SWiRL's RL optimization results — suggesting scale and this style of self-improvement may compound together rather than being independent levers.


# Bringing the Three Directions Together
Three threads run through this notebook's material on pushing self-improvement further:

1. **Diversity in reasoning chains** is necessary for genuine generalization — a single model's outputs, even at high sampling temperature, tend to lack the diversity that made pre-training data itself so effective, and how to reliably generate that diversity remains an open problem (multi-agent fine-tuning is one concrete answer).
2. **Better verification** — especially verification that doesn't bottleneck on human experts — is critical for extending self-improvement into more domains; building good reward models is itself a hard problem, and meta-verification is one way to make automated verification more trustworthy.
3. **Breaking the data-selection bottleneck** — there's only so much task data humans can curate, and letting models propose their own training tasks (as in Absolute Zero) is one promising way to remove that ceiling, provided a reliable way to validate proposed tasks exists.

## An Open Research Question
How far can self-improvement be pushed using **only verifiable domains** (like coding, where automated checking is straightforward) before that benefit generalizes — or fails to generalize — to genuinely **non-verifiable domains**? Could a model become progressively more capable specifically in verifiable areas, in a way that reduces (even if it doesn't eliminate) the need for labeled data elsewhere? This would be a compute-heavy research question to properly test, but a promising direction.


# What Makes a Domain "Non-Verifiable"?
Not every domain lacks verification entirely — some domains just have verification that's too **slow** or too **expensive** to use inside a tight RL training loop. A few examples:
- **Scientific discovery**, **chip design simulations**, or **chemistry experiments** — these might take **days** to produce a single reward signal (running a slow simulation, or physically running a lab experiment), which is far too slow for RL training loops that typically need hundreds or thousands of iterations. Test-time scaling can sometimes tolerate waiting minutes to an hour for a signal, but RL training generally cannot tolerate waiting days per iteration.
- **Genuinely subjective domains** — creative writing and similar tasks resist verification for a different reason: it's inherently hard to define a precise reward function for "quality" when the answer is subjective. A learned reward model can be built to approximate this, but a model can then learn to **exploit (reward-hack)** that approximation if it's even slightly miscalibrated.

## One Practical Fix: Learn a Fast Surrogate Reward Model
Rather than running an expensive simulation or physical experiment for every training step, one approach is to **collect a large batch of that data offline in advance**, and then train a separate, fast **reward model** that predicts the outcome of the expensive process directly from the input — using that learned prediction as the verification signal inside the RL loop instead of the real (slow, expensive) process. The catch: the resulting reward model's accuracy and generality is only as good as the offline data it was trained on, and if that data doesn't cover the space well, the reward model can be meaningfully wrong in ways that mislead training.

## A Concrete Example: Optimizing Compute Kernels
Even in a domain that seems verifiable on the surface — like generating optimized GPU kernels (as in KernelBench-style work) — compiler execution alone tells you whether generated code is *correct*, but assessing detailed **performance profiles** as code complexity grows is a genuinely harder problem than simple pass/fail correctness checking. In practice, tackling this kind of harder verification problem often means **breaking the larger task into smaller subparts**, having the model reason about each piece with access to relevant reference material or a knowledge base, rather than expecting one end-to-end verification signal to handle the whole complex system at once.


# The Cost of Self-Improvement: Why Efficiency Matters Too
Everything covered so far in this notebook is about making models smarter through self-improvement — but all of that comes with a real cost: **more inference**. Repeated sampling, verification loops, multi-agent debate, self-proposed tasks — all of it means running models more, not less. This raises a parallel, equally important question: how do we make that inference **efficient**?


# Intelligence per Watt: Rethinking Where Inference Happens

**Paper:** [arxiv.org/abs/2511.07885](https://arxiv.org/abs/2511.07885)

## The Current "Mainframe Era" of AI
Right now, most LLM usage — ChatGPT, Gemini, and most large models generally — runs in **centralized cloud infrastructure**, not locally. Even open-source models, when they're large, are typically still served from the cloud rather than run on a personal device. Demand for this cloud compute is growing explosively: one cloud provider's AI-driven compute serving grew roughly **1200x in 20 months**; a major GPU maker saw **10x year-over-year growth**; something on the order of **250 gigawatts** of data center capacity is estimated to be needed to keep up, and that figure keeps climbing. Token processing volume for at least one major provider grew from **160 trillion to 1.3 quadrillion** tokens in about eight months — one of the fastest-growing compute demand curves in recent history.

## Two Converging Trends Suggest an Alternative
1. **Most real-world queries don't need frontier-scale models.** Looking at a large-scale sample of real chatbot usage, roughly **77% of requests** are for tasks like practical guidance, information lookup, or writing — tasks that smaller, local models can often handle just as well as the largest frontier models. (Naturally, as chatbots improve, users do ask more complex questions over time — but the bulk of real traffic still skews toward the simpler end of the spectrum.)
2. **Local inference hardware has improved dramatically.** GPU memory on local accelerators has improved roughly **126x since 2012** — modern laptops can now have on the order of 100GB of memory, enough to run very large models locally, especially in quantized form.

Together, these trends raise a real question: **could local inference meaningfully redistribute demand away from centralized cloud infrastructure?**


# A New Metric: Intelligence per Watt
To study this systematically, the paper defines **intelligence per watt (IPW)**: average task accuracy divided by average power draw to solve that task. This captures both **capability** (can a local model — defined here as ≤20B active parameters — actually answer the query correctly?) and **efficiency** (how much useful compute comes out of each watt of power on local hardware?) in one combined metric.

## Study Scope
- **20+ local models** tested (Qwen, GPT-OSS, Gemma 3, and others).
- Both **enterprise-grade accelerators** and **local/consumer accelerators**.
- **1 million real queries**, sourced from real chat traffic plus reasoning benchmarks (Natural Reasoning, MMLU-Pro, SuperGPQA).
- Metrics tracked: accuracy, energy, latency, compute used.
- All of this data and the underlying profiling tools were **open-sourced**.


# Key Findings
1. **Local models are already quite capable, and improving fast.** Since 2023, accuracy on real chat-style queries improved **3.1x** — in just two years. Local models can now correctly address about **88.7%** of the query types studied — a striking figure, given how recently this would have seemed implausible.
2. **Local accelerators still lag enterprise chips in efficiency.** An Apple M4 Max, for example, delivers roughly **1.4–1.5x lower** intelligence-per-watt than an NVIDIA B200. This isn't surprising: enterprise chips like the B200 are purpose-optimized specifically for LLM workloads, while consumer chips are designed for a broader range of tasks — chip designers historically didn't design consumer hardware with the assumption that LLMs would need to run locally at scale.
3. **Overall intelligence efficiency improved roughly 5.3x over two years** — split between **3.1x from better models** and **1.7x from better hardware efficiency**. Both trends are moving in the same direction: models are getting better at solving problems they previously couldn't, and both model and hardware efficiency are compounding together.

**The overall implication:** more and more real-world AI traffic may become addressable by models that run entirely on edge devices — laptops, phones — rather than requiring a round trip to centralized cloud infrastructure.


# Open Directions Going Forward
A few genuinely open research questions, building on everything covered across this notebook series:

## Foundational Understanding of Test-Time Scaling and Synthetic Data
Why does repeated sampling actually surface correct answers the way it does? What's really happening inside the model when this works? And what are the actual best practices for distilling successful trajectories back into a model through fine-tuning? These questions are used constantly in practice (via RL and data collection loops), but the deeper theoretical understanding of *why* they work as well as they do is still incomplete.

## Continual Learning
Humans improve continuously as they solve problems and encounter new experiences — learning is an ongoing, real-time process. Models, by contrast, currently improve mostly through an **offline** cycle: collect a batch of experience/trajectories, then run a separate fine-tuning pass sometime later. This creates a real mismatch between how humans learn and how models currently do. An open question: what would it take to bring positive (and negative) experiences back into a model in a more natural, ongoing way — moving beyond the current asynchronous "generate data, then fine-tune" cycle?

## Infrastructure for High-Throughput, Low-Latency Test-Time Scaling
Test-time scaling techniques — repeated sampling, iterative revision, tool calling, multi-step back-and-forth — look very different from typical single-turn chatbot usage, which most current inference infrastructure is optimized around. As these more compute-intensive, multi-step techniques become mainstream, there's a growing need for **systems and inference-optimization work** specifically suited to them, rather than infrastructure built primarily around simple single-turn exchanges.

## What Got Less Coverage
Pre-training itself received comparatively little direct coverage across this material — the emphasis throughout was mostly on **post-training and test-time scaling methods**, and the newer territory of synthetic data flywheels, continual learning, and the connection between fine-tuning and online/test-time learning. That leaves real open ground for future exploration.


# Practical Directions Following From the Local-Inference Trend
Building on the intelligence-per-watt findings, a few concrete engineering directions follow:
- **Hybrid inference serving:** systems that can smoothly route a given query between local and cloud models/accelerators, depending on the query's complexity and the resources available — rather than sending everything to the cloud by default.
- **Energy-efficient architectures and kernels for local accelerators specifically** — an area that's received comparatively little dedicated attention so far, since most architecture and kernel optimization work has historically targeted cloud-scale accelerators.
- **Energy itself as the target metric.** As compute demand keeps growing, energy is likely to become the most valuable constrained resource — meaning metrics like intelligence-per-watt, and better underlying tools for measuring and optimizing power/energy usage directly, are likely to become far more mainstream targets of optimization than they are today.


# Continual Learning: Weights, Memory, or Context?
Following up on the continual learning question raised earlier in this notebook, there are a few genuinely different ways to approach it — not just one:

## Option 1: Update the Model's Weights
Teaching a model a new skill in a way that's retained going forward — being able to "learn how to learn" (e.g., picking up a new skill from watching a demonstration, rather than needing extensive task-specific demonstrations) — remains a capability current models largely lack. This connects to a broader theme: if the goal is genuine skill transfer (like cross-embodiment generalization in robotics, where a skill learned in one context needs to generalize to a physically different one), adding external memory systems alone doesn't achieve that — actually updating the model's weights tends to be what's required for that kind of transfer.

## Option 2: Rely on (Effectively Infinite) In-Context Learning
If a model could hold and reason over an **effectively infinite context**, remembering and reasoning over everything (positive and negative experiences alike) placed into it, that alone could function as a form of continual learning — without ever touching the model's weights. In practice, this doesn't yet exist: even with contexts in the range of a few million tokens, a model's ability to reason well over everything in that context degrades as it grows.

## Option 3: Long-Context Representations Without Fine-Tuning
A middle path: rather than either fine-tuning the model's weights or relying on raw in-context learning, it's possible to distill a large document corpus into a **compact, reusable KV-cache representation** — a "Cartridge" — that can be loaded at inference time to simulate having that whole corpus in context, without the memory cost of actually keeping it there, and without fine-tuning the underlying model at all. **Paper:** [arxiv.org/abs/2506.06266](https://arxiv.org/abs/2506.06266) (Eyuboglu et al., Stanford) — this approach reportedly matches in-context-learning-level performance on challenging long-context benchmarks while using dramatically less memory and enabling substantially higher throughput.

## Which Approach Is Easiest in Practice?
It depends on the application. If what's needed is essentially a **knowledge base** the model can consult (e.g., "look this fact up"), updating a memory store or database is generally the simpler, more direct solution. But if what's actually needed is teaching the model to **reason well in a genuinely new domain** — not just recall facts, but transfer a skill — a side memory system alone typically doesn't achieve that; updating the model's weights tends to be what's actually required. The robotics cross-embodiment example is a clear illustration: no amount of added memory substitutes for updating the underlying model's weights when the goal is genuine skill transfer.


# Do Agents Need to Self-Create Their Own Environments?
Referring back to Absolute Zero: since that paper's coding environment effectively served as the data source for post-training, is there similar work on agents **creating their own environments**, more generally?

The more useful framing isn't really "is there a paper for self-created environments" — it's: **what real-world task are you actually trying to represent, and can it be simulated well?** In earlier eras of narrow AI, simulated environments for games worked well as training environments precisely because game rules define a **finite, well-specified space** — genuinely easy to represent in code and simulate accurately. Environments matter in current systems specifically because they act as a **proxy for real-world tasks and feedback** — whether an agent or a human builds that simulated environment is almost a secondary detail; what actually matters is whether the resulting environment is a *reasonable, faithful proxy* for how a model will actually interact with the real world when it counts.
